# **Overview**

**Goal**: translate a short Vietnamese audio clip into English text - directly, from one model, with no intermediate Vietnamese transcript.

This is **direct speech-to-text translation**: instead of chaining ASR and machine translation, you train one model to map Vietnamese audio straight to English - harder than either step alone, and a window into how multilingual speech models work.

## **Ground rules**
**Models**: any OpenAI Whisper variant (whisper-tiny → whisper-large-v3, community fine-tunes included). No other families (SeamlessM4T, NLLB, …).
External data: any public ASR/MT dataset (FLEURS, CoVoST 2, VIVOS, CommonVoice, …).
**Not allowed**: paid inference APIs (GPT-4, Gemini, Claude, …) or anything needing internet at scoring time.

## **Description**

### **The task**
Audio in, English out, with no Vietnamese transcript in between: `f : Audio waveform → English string`

This differs from the cascaded approach (ASR → Vietnamese transcript → MT), which is easy to train and debug but lets errors compound - a misheard tone or dropped diacritic in stage 1 leaves stage 2 nothing to recover from. A direct model trains end-to-end on (audio, English) pairs and commits to English phrasing without ever materializing a transcript. Whisper's built-in translate mode is a strong starting point.

### **The data**

2,481 clips, ~9.1 h total. Each 7-15 s, mono, 16 kHz, `.wav`. Sampled from public Vietnamese YouTube (current-affairs and culture commentary), with web-audio conditions: background music, reverb, varied accents, occasional English code-switching.

No training set is provided - all 2,481 clips are the test set. You train on public external corpora and predict on these. You get `sample_submission.csv` (every clip_id) and the `.wav` files at `/kaggle/input/competitions/assignment-2-speech-translation`; the English references are held out for scoring.

### **After the baseline**

**Scale up**: whisper-medium / large-v3 beat small by several BLEU; LoRA still fits Kaggle GPU (2xT4 or 1 P100).

**Better data**: add CoVoST 2, VIVOS, or your own audio/English pairs beyond clean FLEURS.

**Decoding**: tune beam width, length penalty, suppression tokens, no_repeat_ngram_size.

**Ensembling**: average two fine-tuned models' logits.

**Preprocessing**: VAD trimming, loudness normalization, light SpecAugment.

## **Evaluation**

### **Metric**

**Corpus-level BLEU-4** between your English and the hidden references 

- Moses 13a tokenizer, lowercase, no smoothing. Range 0-100, higher is better. BLEU is the geometric mean of clipped n-gram precisions times a brevity penalty that punishes short output:

Scoring is bit-identical to:

`sacrebleu.corpus_bleu(hyps, [refs], lowercase=True, tokenize="13a")`

Run it on a held-out split (e.g. FLEURS Vi→En test) to check scores locally.

**Leaderboard**: the live board reflects 30% of the test set (public); final standings use the held-out 70% (private), revealed at close. Stratified by audio source.

### **Submission**
Submit a Kaggle notebook that loads the test audio, runs your model, and writes `submission.csv` to the working directory with two columns:

```csv
clip_id,prediction
spd_4kn7lnDyxbw_chunk0131_1744305_1758868,the future of our cities will depend on metro lines and well planned satellite towns
```

`clip_id` must match the `.wav` file stems exactly (no extension); use `sample_submission.csv` as your template.

`prediction` is one line of plain English; newlines become spaces.

Exactly one row per `clip_id`. Empty, null, duplicate, or missing rows are rejected.

In [1]:
import torch

# Confirm a GPU is attached - training and inference both need one.
assert torch.cuda.is_available(), \
    "No GPU found. Settings → Accelerator → GPU T4 ×2 (or any GPU)."

# Quick sanity check that CUDA actually works (catches a broken runtime early).
_ = torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")
print(f"torch={torch.__version__} | {torch.cuda.get_device_name(0)}")

torch=2.10.0+cu128 | Tesla T4


**Install the extra packages**. Kaggle already includes PyTorch and Transformers - we just add the few that are missing. `--no-deps` tells `pip` not to reinstall things Kaggle already set up (it stops `pip` from quietly swapping in a different PyTorch and breaking the GPU).

In [2]:
import importlib, subprocess, sys

# datasets<3.0 - the FLEURS loader script was removed in datasets 3.x, so we pin an older version.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "datasets<3.0"])

# peft = the LoRA library. --no-deps so pip doesn't try to swap out Kaggle's PyTorch.
if importlib.util.find_spec("peft") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "peft"])

# evaluate + sacrebleu - used to measure BLEU.
for pkg in ("evaluate", "sacrebleu"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Re-check CUDA still works after installing (a bad install can break torch).
_ = torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")
print("CUDA OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 11.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you have numba-cuda 0.30.2 which is incompatible.
torch 2.10.0+cu128 requires cuda-bindings==12.9.4; platform_system == "Linux", but you have cuda-bindings 13.2.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.8 MB/s eta 0:00:00
CUDA OK


In [3]:
# Standard library + scientific Python
import gc, os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd
import soundfile as sf          # reads .wav audio files
import torch

# Hugging Face: datasets, LoRA (peft), and the Whisper model + Trainer
import evaluate
from datasets import Audio, DatasetDict, load_dataset, load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

In [4]:
# Where files get read from / written to. On Kaggle this is /kaggle/working.
WORK_DIR     = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
FLEURS_CACHE = WORK_DIR / "fleurs_vi_en"        # cached copy of the training data
OUT_DIR      = WORK_DIR / "whisper-large"    # where the model is saved
SUBMISSION   = WORK_DIR / "submission.csv"      # the file Kaggle scores

device = "cuda"
print(f"working dir: {WORK_DIR}")

working dir: /kaggle/working


## 1. Locate the competition test set

The competition dataset is mounted under `/kaggle/input/`. The block below grabs every `test/*.wav` in your attached competition data automatically. Running locally? Make sure the test wavs live in some `<dir>/test/*.wav` structure.

In [5]:
def find_test_dir() -> Path:
    """Find the folder of competition test .wav files under /kaggle/input/.

    The dataset can be attached inside a versioned subfolder, so we look at every
    `*/test` directory and keep whichever one actually holds the most .wav files.
    """
    comp_root = Path("/kaggle/input/competitions/assignment-2-speech-translation")

    candidates = []
    if comp_root.exists():
        candidates = sorted(comp_root.glob("*/test"))

    if not candidates:
        raise FileNotFoundError(
            "No test/ directory found under /kaggle/input/. "
            "Attach the competition dataset via the right-hand sidebar."
        )

    # Pick the candidate folder containing the most .wav files.
    return max(candidates, key=lambda p: len(list(p.glob("*.wav"))))


TEST_DIR  = find_test_dir()
TEST_WAVS = sorted(TEST_DIR.glob("*.wav"))   # every test clip, sorted by filename
print(f"Test dir : {TEST_DIR}")
print(f"Test wavs: {len(TEST_WAVS)}")

Test dir : /kaggle/input/competitions/assignment-2-speech-translation/public_release_v2/test
Test wavs: 2481


## 2. Predict on the competition test set

In [6]:
ZERO_SHOT_MODEL_ID = "openai/whisper-large"

processor = WhisperProcessor.from_pretrained(
    ZERO_SHOT_MODEL_ID,
    language="vi",
    task="translate",
)

model = WhisperForConditionalGeneration.from_pretrained(
    ZERO_SHOT_MODEL_ID,
    torch_dtype=torch.bfloat16,
).to(device)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [7]:
INFER_BATCH      = 8                         # clips translated at once at the end
SAMPLING_RATE    = 16_000                    # Whisper expects 16 kHz audio

In [8]:
# =========================
# Shared decoding config
# =========================

GENERATION_KWARGS = {
    "num_beams": 2,
    "no_repeat_ngram_size": 3,
    # "max_new_tokens": 225,
}

LANGUAGE = "vi"
TASK = "translate"

In [9]:
def load_audio(path: Path) -> np.ndarray:
    """Read one .wav file as a mono, 16 kHz float array (what Whisper expects)."""
    array, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if array.ndim > 1:
        array = array.mean(axis=1)          # stereo → mono
    if sr != SAMPLING_RATE:
        import librosa
        array = librosa.resample(array, orig_sr=sr, target_sr=SAMPLING_RATE)
    return array.astype(np.float32)

**Translate a batch of clips.** A helper that loads each `.wav`, runs the model, and decodes the output ids back into English text - `INFER_BATCH` clips at a time.

In [10]:
def translate_wavs(model, wavs: List[Path]) -> List[str]:
    """Translate a list of .wav files into English strings, INFER_BATCH clips at a time."""
    model.eval()
    forced_ids = processor.get_decoder_prompt_ids(language="vi", task="translate")
    model_dtype = next(model.parameters()).dtype
    outputs: List[str] = []
    for start in range(0, len(wavs), INFER_BATCH):
        batch = wavs[start:start + INFER_BATCH]
        arrays = [load_audio(p) for p in batch]
        inputs = processor.feature_extractor(
            arrays, sampling_rate=SAMPLING_RATE, return_tensors="pt", padding=True
        )
        feats = inputs.input_features.to(device, dtype=model_dtype)
        attn  = inputs.attention_mask.to(device) if "attention_mask" in inputs else None
        with torch.no_grad():
            ids = model.generate(
                feats,
                attention_mask=attn,
                forced_decoder_ids=forced_ids,
                **GENERATION_KWARGS,  # block repeated bigrams
                # max_new_tokens=225,
            )
        outputs.extend(processor.tokenizer.batch_decode(ids, skip_special_tokens=True))
        if (start // INFER_BATCH) % 20 == 0:
            print(f"  …{start + len(batch)}/{len(wavs)}")
    return outputs

**Run it on all the test clips**. This is the slow step - progress prints every ~160 clips.

In [11]:
# Fold the LoRA adapter into the base weights, then translate every competition clip.
#ft_model = model.merge_and_unload().to(device)
print(f"Translating {len(TEST_WAVS)} test clips with beam={GENERATION_KWARGS["num_beams"]}…")
predictions = translate_wavs(model, TEST_WAVS)
print("Done.")

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


Translating 2481 test clips with beam=2…


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

  …8/2481
  …168/2481
  …328/2481
  …488/2481
  …648/2481
  …808/2481
  …968/2481
  …1128/2481
  …1288/2481
  …1448/2481
  …1608/2481
  …1768/2481
  …1928/2481
  …2088/2481
  …2248/2481
  …2408/2481
Done.


Write `submission.csv`
Two columns - `clip_id`, `prediction` - one row per test clip. This is the file Kaggle will score.

In [12]:
# Build the submission: one row per clip, columns clip_id + prediction.
# Any empty prediction is replaced with "_" so every clip has a non-empty entry.
submission = pd.DataFrame({
    "clip_id": [p.stem for p in TEST_WAVS],
    "prediction": [
        p.strip().lower() if p.strip() else "_"
        for p in predictions
    ],
}).sort_values("clip_id").reset_index(drop=True)

submission.to_csv(SUBMISSION, index=False)

print(f"Wrote {len(submission)} predictions to {SUBMISSION}")
print(submission.head())

Wrote 2481 predictions to /kaggle/working/submission.csv
                                    clip_id  \
0  dbcb_A2-Rj67Cf8U_chunk0026_282306_295230   
1  dbcb_Fp0A8LZTC9o_chunk0027_296866_309662   
2  dbcb_Fp0A8LZTC9o_chunk0031_344002_356414   
3  dbcb_Fp0A8LZTC9o_chunk0032_357218_371742   
4  dbcb_Fp0A8LZTC9o_chunk0033_372578_384862   

                                          prediction  
0  the future of traffic in vietnam is not in the...  
1  it could be the confusion sometimes caused by ...  
2  here, life is still going on, people are still...  
3  according to all the dry numbers from internat...  
4  living is not just about staying in a house or...  
